In [1]:
import torch
import torch.nn as nn
import sys
sys.path.append('../../01_Transformer/transformer_pytorch/src')
sys.path.append('./src')

from attention import MultiHeadAttention
from feedforward import FeedForward
from layer_norm import LayerNorm
from encoder import Encoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
from bert import BERT

model = BERT(vocab_size=30000).to(device)

batch, seq_len = 2, 128
x = torch.randint(0, 30000, (batch, seq_len)).to(device)
segment_ids = torch.zeros(batch, seq_len, dtype=torch.long).to(device)
mask = torch.ones(batch, 1, 1, seq_len).bool().to(device)

mlm_output, nsp_output = model(x, segment_ids, mask)

print("MLM output shape:", mlm_output.shape)  # (2, 128, 30000)
print("NSP output shape:", nsp_output.shape)  # (2, 2)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

MLM output shape: torch.Size([2, 128, 30000])
NSP output shape: torch.Size([2, 2])
Total parameters: 132,745,010


In [ ]:
from datasets import load_dataset
from tokenizer import Tokenizer
from bert_dataset import get_bert_dataloader

dataset = load_dataset("opus100", "en-ko")

en_tokenizer = Tokenizer()
en_tokenizer.load('./data/processed/en_tokenizer.model')

train_loader = get_bert_dataloader(
    dataset['train'],
    tokenizer=en_tokenizer,
    batch_size=32,
    max_seq_len=128
)

# Check one batch
tokens, segment_ids, mlm_labels, nsp_labels, mask = next(iter(train_loader))

print("tokens shape    :", tokens.shape)       # (32, max_len)
print("segment shape   :", segment_ids.shape)  # (32, max_len)
print("mlm_labels shape:", mlm_labels.shape)   # (32, max_len)
print("nsp_labels shape:", nsp_labels.shape)   # (32,)
print("mask shape      :", mask.shape)          # (32, 1, 1, max_len)